# 07 Shell 脚本与自动化（Bash）

> 覆盖范围：Shell 是什么（与终端/内核的关系）、变量与引号、条件与循环、函数、管道与重定向、通配符与正则区别、字符串与数组处理、常用脚本模式（监控/批量处理/部署）、`set -e` 健壮性、bash vs zsh。
> 使用方式：先读**原理讲解**（shell 如何解释命令、引号规则、管道数据流），再看**脚本示例**（默写），最后用 Python 模拟核心语义验证，自测面试追问。
> 注：本 notebook 内核为 Python 3；Shell 片段用于阅读与默写，语义对照实验用 Python 模拟以保持可执行验证。

---

## 1. Shell 是什么

- **Shell = 命令行解释器**：读入你敲的命令 → 解析 → 调用系统调用执行 → 等待并显示结果；
- 你不是在和内核对话，而是和 shell（bash/zsh）对话，shell 再转给内核；
- **脚本（script）** 就是把一串命令 + 逻辑写进文件，让 shell 逐行执行（`#!/bin/bash` 指定解释器，`bash script.sh` 运行）。

```bash
#!/bin/bash          # shebang：告诉系统用什么解释器
echo "Hello, $USER" # 双引号内变量展开
```

## 2. 变量与引号（最易错）

| 写法 | 含义 | 示例 |
| --- | --- | --- |
| `name="world"` | 赋值（= 两侧**不能有空格**！） | `name="alice"` |
| `$name` / `${name}` | 取值 | `echo "hi $name"` |
| `"$var"` | **双引号**：变量展开，保留空格 | `echo "$name"` → 一个参数 |
| `'$var'` | **单引号**：字面量，不展开 | `echo '$name'` → 输出 $name |
| 反引号 / `$(...)` | 命令替换 | `now=$(date)` |
| `$?` `$0` `$1` `$#` `$@` | 上次退出码 / 脚本名 / 位置参数 / 参数个数 / 全部参数 | `if [ $? -eq 0 ]` |

> ⚠️ **易错点**：`if [ $var = "x" ]` 中 `[` 是**命令**（test），左右必须空格；`[ $var = "x" ]` 变量一定要加引号，否则空值或含空格会语法错误/分裂。

## 3. 条件与循环

```bash
# if
if [ -f /etc/passwd ]; then echo "存在"; fi   # -f 文件存在 -d 目录存在 -z 空串
if [ "$age" -gt 18 ]; then echo adult; else echo minor; fi  # 数字比较 -gt -lt -eq
if [ "$name" = "alice" ]; then echo hi; fi                 # 字符串比较 = !=

# for
for f in *.txt; do echo "处理 $f"; done
for i in $(seq 1 5); do echo $i; done
# C 风格
for ((i=0; i<5; i++)); do echo $i; done

# while / until
i=0; while [ $i -lt 5 ]; do echo $i; i=$((i+1)); done
# 逐行读文件
while read -r line; do echo "$line"; done < data.txt

# case
case "$1" in
  start) echo starting;;
  stop)  echo stopping;;
  *)     echo unknown;;
esac
```

## 4. 函数与脚本参数

```bash
greet() {                       # 定义函数（function 关键字可省略）
    local name="$1"           # local 局部变量
    echo "hello $name"
    return 0                    # 返回码
}
greet "$1"                     # 调用

# 处理全部参数
for arg in "$@"; do echo "$arg"; done
shift                           # 丢弃第一个位置参数
```

## 5. 管道与重定向（核心数据流）

```
3 个标准流：stdin(0) / stdout(1) / stderr(2)

重定向：
  >   覆盖写 stdout（先清空）        >>  追加写
  2>  重定向 stderr                  2>&1  stderr 并到 stdout
  <   从文件读 stdin                  <<EOF heredoc 多行输入
  &>  全部（bash 4+）

管道：cmd1 | cmd2  =  cmd1 的 stdout 接到 cmd2 的 stdin（并发执行）
```

```bash
python train.py > log.txt 2>&1 &    # 日志写入文件，后台运行
ps -ef | grep python | grep -v grep | awk '{print $2}' | xargs kill -9  # 杀进程链
cat <<EOF > config.yaml            # heredoc 写文件
key: value
EOF
```

## 6. 通配符 vs 正则表达式（两者常混）

| | 通配符（文件名展开，shell 做） | 正则表达式（文本匹配，grep -E 等做） |
| --- | --- | --- |
| 匹配位置 | 匹配**文件名** | 匹配**文本内容** |
| `*` | 任意串 | 前一字符重复 0 次以上 |
| `?` | 单字符 | 前一字符 0 或 1 次 |
| `[abc]` | 字符类 | 字符类（相同） |
| 其他 | `[!a]` 排除 | `^` 行首、`$` 行尾、`+` `{n}` 等 |

## 7. 数组与字符串（bash）

```bash
arr=(a b c)               # 数组
echo ${arr[0]} ${arr[@]}  # a a b c
echo ${#arr[@]}           # 长度 3

s="hello world"
echo ${#s}                # 长度 11
echo ${s:0:5}             # 子串 hello
echo ${s//o/0}            # 全部替换 hello→hell0
echo ${s/world/there}     # hello there
```

## 8. 健壮脚本：set 系列

```bash
set -e     # 任何命令失败（非 0）立即退出——脚本不静默继续
set -u     # 使用未定义变量报错
set -o pipefail  # 管道中任一命令失败则整体失败（配合 -e）
set -euo pipefail   # 生产脚本标准三件套
```

## 9. bash vs zsh

- **bash**：默认 Linux shell、POSIX 兼容、写**脚本**用它（可移植）；
- **zsh**：交互体验好（补全强、插件 oh-my-zsh），脚本兼容性略差；
- 生产环境脚本一律 `#!/bin/bash`。

## 10. 进阶：参数展开全家族（面试高频）

`${var}` 只是最基础写法，bash 的参数展开是一整套语法，写健壮脚本必须掌握：

| 写法 | 含义 | 示例与结果 |
| --- | --- | --- |
| `${var:-默认}` | 未定义/为空 → 用默认值（**不**修改 var） | `echo ${PORT:-8080}` |
| `${var:=默认}` | 未定义/为空 → 赋值默认再输出（**修改** var） | 初始化惯用法 |
| `${var:?报错}` | 未定义/为空 → 报错退出（防呆） | `${MODE:?需要指定 MODE}` |
| `${#var}` | 字符串长度 | `echo ${#name}` → 5 |
| `${var:0:5}` | 子串（从 0 取 5 个） | `${s:1:3}` 切片 |
| `${var#前缀}` | 去**最短**匹配前缀 | `${path#*/}` |
| `${var##前缀}` | 去**最长**匹配前缀 | `${path##*/}` → 取最后一段（basename 技巧） |
| `${var%后缀}` | 去**最短**匹配后缀 | `${path%.txt}` 去扩展名 |
| `${var/旧/新}` | 替换**第一个** | `${s/a/b}` |
| `${var//旧/新}` | 替换**全部** | `${s// /_}` 空格换下划线 |

```bash
path="/home/alice/proj/run.sh"
echo ${path##*/}      # run.sh        （取文件名，等同 basename）
echo ${path%/*}       # /home/alice/proj （取目录名）
echo ${path%.sh}      # /home/alice/proj/run
file="a b c.txt"
echo ${file// /_}     # a_b_c.txt     （空格替换为下划线）
```

## 11. find 与 xargs：批量处理文件

```bash
# find：按条件搜索（比 ls 递归灵活得多）
find /data -name "*.log" -mtime +7       # 7 天前修改的 .log
find /data -type f -size +100M            # 大于 100M 的文件
find /data -name "*.tmp" -delete         # 找到并删除
find /data -name "*.log" -exec rm {} \;  # 找到并对每个执行 rm（{} 是占位符）

# xargs：把前一条输出转成后一条的参数（处理大量文件时避免参数过长）
find /data -name "*.log" | xargs rm       # 等价上面，按行转参数
find /data -name "*.log" -print0 | xargs -0 rm   # 处理含空格/换行的文件名（安全做法）
echo a b c | xargs -n 1 echo              # -n 1：每个参数单独执行一次
```

> ⚠️ **易错点**：`rm $(find ...)` 在文件名含空格时会分裂；`find ... | xargs` 也危险。**安全姿势**：`find -print0 | xargs -0` 或 `find -exec {} +`。

## 12. `[ ]` 与 `[[ ]]`（bash 专属）

- `[ ... ]`：POSIX 的 test 命令——`[` 是一个程序，参数要空格分隔，空值/含空格变量**必须加引号**；
- `[[ ... ]]`：bash 内建关键字——**更安全**：变量不加引号也不会分裂，支持 `&&`/`||`、正则 `=~`、`< >` 比较：

```bash
if [[ "$name" == a* ]]; then echo "a 开头"; fi      # 通配匹配（模式）
if [[ $age =~ ^[0-9]+$ ]]; then echo "纯数字"; fi    # 正则 =~
if [[ -n "$x" && -f "$x" ]]; then ...; fi          # 逻辑组合
```

## 13. `$@` 与 `$*` 的区别（极易考）

- `"$@"`：把每个参数**分别**当作一个参数（保留引号语义）——**推荐**；
- `"$*"`：把所有参数**合并成一个字符串**（空格分隔）；
- 不加引号的 `$*`/`$@` 都会做词分裂（危险）。

```bash
show() { for x in "$@"; do echo "[$x]"; done; }
show "a b" c    # [a b]  [c]   （"$@" 保住了含空格的 a b）
show "$*" 的语义 = 把 "a b" 和 c 拼成 "a b c"（一个参数）
```

## 14. heredoc 的细节

- `<<EOF ... EOF`：内容中**变量会展开**；
- `<<'EOF' ... EOF`：**字面量**，不展开（写代码/模板时用）；
- `<<-EOF`：允许缩进（行首 Tab 被忽略）；
- 结束符必须顶格写，前面不能有空格。

```bash
cat <<'PY' > script.py          # 字面写入，$ 不解释
print("cost: $", 100)
PY
```

## 15. cron 定时任务

```
# 分 时 日 月 周
0 2 * * *    /opt/backup.sh          # 每天 2:00 执行备份
*/10 * * * * /opt/check.sh > log     # 每 10 分钟
0 9 * * 1-5  /opt/job.sh             # 工作日 9:00
30 23 * * 6  /opt/weekly.sh          # 周六 23:30
```

- `crontab -e` 编辑、`crontab -l` 查看；
- 环境变量少（没有你家 shell 的环境）——脚本内要写全路径、`PATH=/usr/bin:/bin`；
- 日志重定向到文件（`>> log 2>&1`），否则出错无迹可查。


## 核心概念

### (a) 🗂️ Shell：从「词法展开」到「自动化脚本」的主线

Shell 脚本的本质是**把命令串起来**：先理解变量怎么展开、引号怎么区分，再理解管道怎么把数据从上一个命令传给下一个，最后把这些组合成能自动干活的脚本。

```mermaid
flowchart TB
    A["🧠 Shell 词法<br/>变量展开 / 引号 / 命令替换"]
    A --> B["🔀 管道流水线<br/>A | B | C 逐级传递"]
    B --> C["📜 脚本模式<br/>批量重命名 / 监控告警"]
    C --> D["💬 返回值与默认参数<br/>$? / ${var:-默认}"]
    D --> E["🎯 面试 30 秒速答<br/>默写高频命令与陷阱"]
```

- **Shell 词法**：双引号内变量会展开、单引号内原样输出、反引号或 `$()` 做命令替换——写脚本出错十有八九是引号问题。
- **管道流水线**：`A | B | C` 把 A 的 stdout 接到 B 的 stdin，数据逐级流动、前一个命令看不到后一个的输出；`grep -v` 反向过滤也常配合管道。
- **脚本模式**：批量重命名（for 循环 + mv）、磁盘监控告警（df + 阈值判断）是最高频的两个场景，面试常让现场写。
- **返回值与默认参数**：`$?` 拿上一条命令退出码、`${var:-默认}` 给变量兜底、`set -e` 遇错即停——脚本健壮性的三个关键点。
- **面试 30 秒速答**：能把每条命令的用途、常见参数、易错点（引号/管道/返回值）默写出来，才算过关。

## 本节要解决的问题

| 要解决的问题 | 对应内容 |
|---|---|
| 变量加了引号怎么就不展开了？ | 引号规则与变量展开 |
| 命令里再嵌一条命令怎么写？ | 命令替换 `$()` |
| `A \| B \| C` 数据怎么流？ | 管道与重定向 |
| `*.py` 和 `.*` 有什么不同？ | 通配符 vs 正则 |
| 脚本怎么自动重复干活？ | 条件与循环 |
| 脚本一崩就全线崩怎么办？ | `set -e` 与返回值 |

## 故事引入

你每天下午都要做同一件烦人事：把 200 个 `report_2024xxxx.csv` 重命名成 `daily_xxxx.csv`，再把磁盘告警邮件里的 IP 去重统计。手工做要一小时，还老出错。于是你写了个 Shell 脚本：

```bash
for f in report_*.csv; do mv "$f" "daily_${f#report_}"; done
df -h | awk 'NR>1 && $5+0>80 {print $6, $5}' > alert.txt
```

第一次跑就踩坑：`mv $f ...` 没加引号，文件名带空格直接裂开；把 `*.csv` 写进双引号又变成字面量。你一边修一边明白了：Shell 不是「翻译器」，而是一套**先展开、再执行**的词法规则——引号、变量、通配符、管道各有各的时序。这篇把 Shell 的「词法与数据流」讲透，你就能写出健壮到能交班的自动化脚本。

## 深入原理

### ① 变量展开与引号三兄弟

**直觉**：Shell 读一行命令时先做**展开（expansion）**：`$var`、`*.csv`、`$(cmd)` 会在执行前被替换成实际内容。引号是「展开开关」：**双引号**内变量仍展开、**单引号**内一切原样、**无引号**时还会做单词分割与通配符展开。

**要点**：`x="a b"; echo $x` 输出 `a b`（被拆成两个参数）而 `echo "$x"` 仍是一个参数——这正是「文件名带空格要加引号」的原因。

**面试怎么问**：「$var 和 "$var" 和 '$var' 区别？」——分别：展开+分割 / 展开不分割 / 完全字面量。

![Shell 引号、变量展开与分词规则](images/shell_quote.svg)

> 📌 引号三兄弟：双引号展开变量、单引号全字面量、无引号还分词+通配

### ② 命令替换 `$()` 与反引号

**直觉**：想「先用一条命令的结果，再喂给下一条」，用命令替换：`$(date)` 会先执行 `date`，把输出当文本填入。老写法用反引号 `` `cmd` ``，新写法更推荐 `$(cmd)`——可以嵌套、可读性好。

**要点**：`echo "今天: $(date +%F)"`；`now=$(date +%s)` 拿时间戳。注意命令替换在**双引号内也会执行**（它发生在变量展开之前）。

**面试怎么问**：「$() 和反引号哪个好、为什么？」——`$()`：支持嵌套、转义更简单；反引号嵌套要 `\`，易错。

### ③ 管道与重定向：数据流

**直觉**：**管道** `A | B` 把 A 的标准输出（stdout）接到 B 的标准输入（stdin），数据**逐条流动**而不是攒完再给——所以 `tail -f log | grep ERR` 能实时过滤。**重定向** `>`/`>>`/`<` 把流向改到文件。

**要点**：`2>` 重定向标准错误、`2>&1` 合并、`|&` 也合并管道。管道里每个命令是**并发子进程**，慢的环节决定整体速度。

**面试怎么问**：「管道和重定向本质区别？」——管道是进程间内存数据流；重定向是把 fd 指向文件/设备。

![Shell 管道 stdin、stdout 与 stderr](images/shell_pipe.svg)

> 📌 管道数据逐条流动且各命令并发执行，stderr 用 2> 单独重定向

### ④ 通配符 vs 正则表达式

**直觉**：**通配符**是 Shell 在展开阶段用的「文件匹配」：`*` 任意串、`?` 单字符、`[ab]` 字符集——它**匹配文件名**。**正则**是 grep/sed/awk 里匹配**文本内容**的语法：`.*` 任意串、`^` 行首、`+` 一次以上。

**要点**：`*.py`（Shell 找文件）vs `grep -E '^import .*' file.py`（正则匹配行内容）。易错：正则的 `*` 是「重复前一个字符」而不是「任意串」。

**面试怎么问**：「rm *.txt 和 grep '*.txt' 的 * 一样吗？」——前者通配符（匹配文件名），后者正则里 `*` 是量词，实际应写 `\.txt$`。

![Shell glob 与 regex 对比](images/shell_glob_regex.svg)

> 📌 通配符匹配文件名（Shell 展开阶段），正则在 grep/sed/awk 里匹配行内容

### ⑤ 条件与循环：脚本的骨架

**直觉**：`if`/`for`/`while` 让脚本能「看情况干活」和「批量干活」。`if` 判断的是**命令退出码**（0=真）而非布尔表达式：`if grep -q foo file; then ...`。

**要点**：`for f in *.csv` 遍历；`while read line; do ...; done < file` 逐行处理；`[ ]`/`[[ ]]` 做比较（`[[ ]]` 支持 `&&`、正则 `=~`）。数字比较用 `-eq`/`-gt`（因为 `>` 会被当重定向）。

**面试怎么问**：「为什么 if 里用 [ 而不是比较运算符？」——`[` 是 test 命令的别名，返回退出码；`[[ ]]` 是 bash 内建，更安全。

### ⑥ `set -e` 与脚本健壮性

**直觉**：默认 Shell 脚本**遇错不退出**——下一条命令照样跑，错误会被悄悄吞掉。生产脚本要加 `set -e`（一出错就停）、`set -u`（用未定义变量报错）、`set -o pipefail`（管道里任一段失败都算失败）。

**要点**：`$?` 拿上一条命令退出码；`${var:-默认}` 给变量兜底；`trap 'cleanup' EXIT` 保证退出时清理临时文件。

**面试怎么问**：「set -e 的坑？」——它只对「未在条件上下文」的失败生效：`if cmd; then` 里的 cmd 失败不算。所以常配 `|| true` 或 `set -o pipefail`。

In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
# 模拟 Shell 词法：变量展开 + 单双引号差异 + 命令替换（理解引号规则）

USER = 'alice'
env = {'USER': USER, 'HOME': '/home/alice'}

def expand(token, env, single_quote=False):
    """模拟 bash 的引号语义：双引号内展开 $var，单引号内不展开"""
    if single_quote:
        return token  # 单引号：字面量
    out = []
    i = 0
    while i < len(token):
        if token[i] == '$':
            j = i + 1
            name = ''
            while j < len(token) and (token[j].isalnum() or token[j] == '_'):
                name += token[j]; j += 1
            out.append(env.get(name, ''))
            i = j
        else:
            out.append(token[i]); i += 1
    return ''.join(out)

print('双引号: ', expand('Hello, $USER', env))      # 展开
print('单引号: ', expand('Hello, $USER', env, True))  # 字面
print('$$ 后跟普通词: ', expand('$HOME', env))       # /home/alice

# $? 上次退出码模拟 + && / || 短路
def run(cmd):
    ok = cmd[0] == 'ok'
    return 0 if ok else 1

code = run(['bad'])
print('失败后 $?:', code)
print('cmd1 && cmd2 短路:', '执行cmd2' if code == 0 else '跳过cmd2')
print('cmd1 || cmd2 兜底:', '执行兜底' if code != 0 else '不动')
assert expand('Hello, $USER', env) == 'Hello, alice'
assert expand('Hello, $USER', env, True) == 'Hello, $USER'

In [ ]:
# 模拟管道数据流：A | B | C 逐级传递 + grep -v 过滤（理解流水线）

def pipe(*commands):
    """按顺序把上一个输出喂给下一个输入"""
    data = None
    for cmd in commands:
        data = cmd(data)
    return data

# 三个模拟命令
def ps_lines(_):
    return ['PID  COMMAND', '100 python', '101 grep', '102 python', '103 sshd']

def grep_python(data):
    return [l for l in data if 'python' in l]

def grep_v_grep(data):
    return [l for l in data if 'grep' not in l]   # grep -v grep

def awk_pid(data):
    return [l.split()[0] for l in data]           # awk '{print $1}'

result = pipe(ps_lines, grep_python, grep_v_grep, awk_pid)
print('ps | grep python | grep -v grep | awk → PID:', result)
assert result == ['100', '102'], '应筛出两个 python 进程 PID'

# 重定向语义：> 覆盖 / >> 追加
log = []
def redirect_gt(parts):
    return parts                       # > 覆盖（此处模拟清空）

log = ['t1']
log = ['t2']                           # >> 追加等价：log += ['t2']
print('重定向后日志:', log)

# heredoc 写多行
content = '\n'.join(['key: value', 'nested:', '  a: 1']) + '\n'
print('heredoc 内容:')
print(content, end='')
assert 'key: value' in content

In [ ]:
# 模拟常见脚本模式：批量重命名 + 磁盘监控告警（默认参数 + 返回值）

def batch_rename(files, old, new):
    """模拟 for f in *.txt; do mv $f ${f//old/new}; done"""
    out = []
    for f in files:
        if old in f:
            out.append((f, f.replace(old, new)))
    return out

files = ['a.txt', 'b.txt', 'a.log', 'c.txt']
renamed = batch_rename(files, '.txt', '.md')
for a, b in renamed:
    print(f'mv {a} {b}')
assert [b for _, b in renamed] == ['a.md', 'b.md', 'c.md']


# 磁盘监控：df -h | awk 判断使用率 > 阈值
disks = [
    ('/dev/sda1', 45),
    ('/dev/sdb1', 88),
    ('/tmp/none', 12),
]

def check_disk(disks, threshold=80):
    alerts = [(d, u) for d, u in disks if u > threshold]
    if alerts:
        for d, u in alerts:
            print(f'[告警] {d} 使用率 {u}% > {threshold}%')
        return 1          # 脚本退出码非 0
    print('磁盘正常')
    return 0

code = check_disk(disks)
print('监控脚本退出码:', code, '(非0=告警，可接 CI/告警系统)')
assert code == 1


# set -euo pipefail 模拟
def run_with_set_e(commands):
    for cmd in commands:
        if cmd() != 0:
            print('set -e: 命令失败，立即退出，不再执行后续')
            return 'ABORTED'
    return 'DONE'

def ok(): return 0
def bad(): return 1

print(run_with_set_e([ok, ok]))
print(run_with_set_e([ok, bad, ok]))   # 第三个不会执行

In [ ]:
# 模拟参数展开全家族：${var:-默认} / ${#var} / ## 去前缀 / % 去后缀 / //替换

env = {'path': '/home/alice/proj/run.sh', 'PORT': ''}   # 全局环境（函数默认回退到它）

def param_expand(var, mode, arg=None, env=None, default=None):
    """模拟 bash 常用参数展开（部分）；不传 env 时回退到全局 env"""
    env = env or globals().get('env', {})
    v = env.get(var, '')
    if mode == ':-':
        return v if v else default              # ${var:-默认}
    if mode == '#':
        return len(v)                           # ${#var}
    if mode == '##':
        idx = v.rfind(arg)                      # 最长前缀：取最后出现处之后
        return v[idx + len(arg):] if idx >= 0 else v
    if mode == '%':
        idx = v.rfind(arg)                      # 最短后缀：取最后出现处之前
        return v[:idx] if idx >= 0 else v
    if mode == '//':
        return v.replace(arg, default)          # 全部替换
    raise ValueError(mode)

path = '/home/alice/proj/run.sh'
env = {'path': path, 'PORT': ''}
print('${PORT:-8080}  =', param_expand('PORT', ':-', default='8080'))   # 未定义→默认
print('${#path}       =', param_expand('path', '#'))                    # 长度
print('${path##*/}    =', param_expand('path', '##', '/'))              # run.sh
print('${path%/*}     =', param_expand('path', '%', '/'))               # /home/alice/proj
print('${path//run/start} =', param_expand('path', '//', 'run', default='start'))

assert param_expand('path', '##', '/') == 'run.sh'
assert param_expand('path', '%', '/') == '/home/alice/proj'
assert param_expand('PORT', ':-', default='8080') == '8080'
assert param_expand('path', '#') == len(path)


# 模拟 find -name + xargs：把匹配结果逐行作为参数
def find_files(files, pat):
    return [f for f in files if pat in f]

def xargs_exec(items, template):
    """模拟 xargs -I{} command {}：每个 item 替换 {} 执行一次"""
    return [template.replace('{}', it) for it in items]

files = ['a.log', 'b.log', 'c.txt', 'd.LOG']
logs = find_files(files, '.log')
print('find -name \"*.log\":', logs)
moves = xargs_exec(logs, 'mv {} /tmp/logs/')
for m in moves: print(' ', m)
assert moves == ['mv a.log /tmp/logs/', 'mv b.log /tmp/logs/']

## 16. 面试 30 秒速答

- **Shell = 命令解释器**：解析命令 → 调系统调用；脚本 = 命令 + 逻辑；
- **引号**：双引号展开变量、单引号字面量、`$(...)` 命令替换；赋值 = 两边无空格；
- **条件**：`[` 是 test 命令，左右有空格；数字 `-gt`，字符串 `=`；
- **管道/重定向**：`|` 接 stdout→stdin（并发）；`>` 覆盖 `>>` 追加 `2>&1` 并 stderr；
- **通配符 vs 正则**：通配符匹配文件名、`*`=任意串；正则匹配文本内容、`*`=重复；
- **健壮性**：`set -euo pipefail`；变量引用加引号；local 变量；
- **bash vs zsh**：脚本用 bash（可移植），交互用 zsh（体验好）。
- **参数展开**：`${var:-默认}` 兜底、`${#var}` 长度、`${var##*/}` 取 basename、`${var//a/b}` 全替换；
- **find/xargs**：`find -name -type -mtime -size` 条件搜索；`xargs` 把输出转参数；安全姿势 `-print0 | xargs -0`；
- **`[[ ]]` vs `[ ]`**：`[[ ]]` 是 bash 关键字、支持 `&&`/正则 `=~`、变量免引号；`[ ]` 是 test 命令必须加引号；
- **`$@` vs `$*`**：`"$@"` 逐参数独立（推荐）；`"$*"` 合并成一串；
- **cron**：分时日月周五段；`crontab -e/-l`；注意环境变量与日志重定向。

## 17. 自测清单（面试追问）

- [ ] 单引号与双引号区别？`$(...)` 与反引号区别？什么时候必须加引号？
- [ ] 手写：脚本接收一个目录参数，统计其中 .py 文件行数总和。
- [ ] `>` / `>>` / `2>&1` / `<<EOF` 各是什么？管道符 `|` 与重定向的区别？
- [ ] `$?` / `$0` / `$1` / `$#` / `$@` 分别是什么？`shift` 干什么？
- [ ] 通配符 `*` 与正则 `*` 有什么不同？`grep` 用哪个？
- [ ] `set -e` / `set -u` / `pipefail` 各防什么？为什么生产脚本要 `set -euo pipefail`？
- [ ] 手写：查找所有大于 100MB 的文件并删除（find + xargs/exec）；
- [ ] `[ "$a" = "x" ]` 里为什么变量要加引号？`[ $a = x ]` 会怎样？
- [ ] 手写：每分钟检查某端口是否被占用，占用则重启服务（cron + 脚本）。
- [ ] bash 数组怎么定义？`${arr[@]}` / `${#arr[@]}` / `${s//a/b}` 各是什么？
- [ ] `${var:-默认}` / `${var:=默认}` / `${var:?报错}` / `${#var}` 各是什么语义？
- [ ] `find -name "*.log" -mtime +7` 与 `find . -size +100M -exec rm {} \;` 什么意思？
- [ ] `find ... | xargs rm` 有什么隐患？为什么用 `-print0 | xargs -0`？
- [ ] `[[ $a =~ regex ]]` 与 `[ "$a" = x ]` 的区别？
- [ ] 手写：找出 /tmp 下 7 天内修改的 .log 并按行数排序输出。

> ✅ 全部能回答 + 能默写脚本，本课通过。

## 面试考点（Q&A）

**Q1：写命令把当前目录所有 .log 文件备份成 .bak？**
A：`for f in *.log; do cp "$f" "${f%.log}.bak"; done`。💡 追问：`${f%.log}` 是什么？——去掉后缀的参数展开。

**Q2：`$?` 什么时候用？**
A：拿上一条命令退出码：`cmd; if [ $? -eq 0 ]; then ...`。💡 追问：`$!` 呢？——上一条后台命令的 PID。

**Q3：如何实时过滤日志里的 ERROR 并统计次数？**
A：`tail -f app.log | grep ERROR | wc -l`（或先 grep 再 uniq -c）。💡 追问：为什么能实时？——管道逐条流动不缓存。

**Q4：`a=1; a=2 echo $a` 输出什么？**
A：输出 1——展开发生在赋值生效**之前**，同一行内 `$a` 用的还是旧值。💡 追问：怎么在同一行用新值？——`a=2; echo $a` 分两行，或 `a=2 sh -c 'echo $a'`。

**Q5：脚本默认遇错继续跑，如何让它「一错就停」？**
A：开头加 `set -e`；再加 `set -u`（未定义变量报错）、`set -o pipefail`。💡 追问：set -e 为什么有时没用？——if 条件里的失败不算错，需 `|| true` 等。

**Q6：把 find 找到的文件批量改权限？**
A：`find . -name '*.sh' -exec chmod +x {} \;` 或 `while read` 循环。💡 追问：`{}` 和 `\;` 含义？——{} 是占位符，\; 是 -exec 结束符。

## 小结与下一步

Shell 的威力来自「**展开 → 执行 → 数据流**」三条规则：引号控制展开、管道控制流向、退出码控制分支。记住三个高频场景（批量处理、日志过滤、监控告警）的套路，脚本就能写对、写稳。

**下一步**：进入 08-Go——如果 Shell 是「串命令」，Go 就是「串协程」，用更现代的方式解决并发问题。